# Compute Multi-Tool Fidelity

Runs all 5 tools on all 42 RDS datasets (RDS->H5AD).
Computes 7 fidelity metrics comparing each tool's H5AD output
against CrossCell's H5AD as reference.

**Output**: `benchmark/results/multi_tool_fidelity.json`

## 1. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json, os, re, subprocess, time, traceback
from pathlib import Path
import numpy as np
import anndata as ad
import scipy.sparse as sp
from scipy.stats import pearsonr, spearmanr
import h5py

RESULTS_DIR = Path('/benchmark/results')
DATA_DIR = Path('/benchmark/data/generated')
TMP = Path('/tmp/fidelity_work')
TMP.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = RESULTS_DIR / 'multi_tool_fidelity.json'
ROBUSTNESS_FILE = RESULTS_DIR / 'robustness_benchmark.json'

ALL_TOOLS = ['CrossCell', 'Zellkonverter', 'anndataR', 'convert2anndata', 'easySCF']
OTHER_TOOLS = ['Zellkonverter', 'anndataR', 'convert2anndata', 'easySCF']

print('Setup complete')


Setup complete


## 2. Load Robustness Data & Build Dataset List

In [2]:
with open(ROBUSTNESS_FILE) as f:
    robustness = json.load(f)

# Build success lookup: {tool: {test_id: True/False}}
success_map = {}
for tool_key in ['crosscell', 'zellkonverter', 'anndataR', 'convert2anndata', 'easySCF']:
    tool_name = {'crosscell': 'CrossCell', 'zellkonverter': 'Zellkonverter'}.get(tool_key, tool_key)
    success_map[tool_name] = {}
    for entry in robustness.get(tool_key, {}).get('rds_to_h5ad', []):
        success_map[tool_name][entry['test_id']] = (entry['status'] == 'success')

# Build dataset list from CrossCell's rds_to_h5ad entries
DATASETS = []
for entry in robustness['crosscell']['rds_to_h5ad']:
    DATASETS.append({
        'test_id': entry['test_id'],
        'file': entry['file'],
        'dataset': entry['dataset'],
        'seurat_version': entry.get('seurat_version', ''),
        'type': entry.get('type', ''),
        'n_cells': entry.get('n_cells', 0),
        'n_genes': entry.get('n_genes', 0),
        'nnz': entry.get('nnz', 0),
    })

print(f'Loaded {len(DATASETS)} datasets')
for tool in ALL_TOOLS:
    n_ok = sum(1 for v in success_map[tool].values() if v)
    print(f'  {tool}: {n_ok}/{len(success_map[tool])} succeeded in robustness test')

Loaded 42 datasets
  CrossCell: 42/42 succeeded in robustness test
  Zellkonverter: 42/42 succeeded in robustness test
  anndataR: 42/42 succeeded in robustness test
  convert2anndata: 19/42 succeeded in robustness test
  easySCF: 42/42 succeeded in robustness test


## 3. Helper Functions

In [3]:
def run_crosscell(input_path, output_path, fmt='anndata'):
    r = subprocess.run(
        ['crosscell', 'convert', '-i', input_path, '-o', output_path, '-f', fmt],
        capture_output=True, text=True, timeout=600
    )
    return r.returncode == 0, r.stderr

def run_r_tool(rds_path, h5ad_path, tool_name):
    scripts = {
        'Zellkonverter': (
            'suppressPackageStartupMessages({library(zellkonverter);library(Seurat);library(SingleCellExperiment)})\n'
            f'obj <- readRDS("{rds_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'sce <- as.SingleCellExperiment(obj)\nwriteH5AD(sce, "{h5ad_path}")'
        ),
        'anndataR': (
            'suppressPackageStartupMessages({library(anndataR);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'if (file.exists("{h5ad_path}")) file.remove("{h5ad_path}")\n'
            f'ad <- as_AnnData(obj)\nwrite_h5ad(ad, "{h5ad_path}")'
        ),
        'convert2anndata': (
            'suppressPackageStartupMessages({library(convert2anndata);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'sce <- convert_seurat_to_sce(obj)\nad <- convert_to_anndata(sce)\n'
            f'anndata::write_h5ad(ad, "{h5ad_path}")'
        ),
        'easySCF': (
            'suppressPackageStartupMessages({library(easySCFr);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'saveH5(obj, "{h5ad_path}")'
        ),
    }
    if tool_name not in scripts:
        return False, f'Unknown tool: {tool_name}'
    r = subprocess.run(
        ['Rscript', '-e', scripts[tool_name]],
        capture_output=True, text=True, timeout=600
    )
    return r.returncode == 0, r.stderr

def get_X_matrix(adata):
    """Extract expression matrix from AnnData, handling X=None (anndataR puts data in layers)."""
    if adata.X is not None:
        return adata.X
    # anndataR often stores data in layers instead of X
    if adata.layers:
        # Prefer 'counts' or 'data' layer
        for key in ['counts', 'data', 'X']:
            if key in adata.layers:
                return adata.layers[key]
        # Fall back to first available layer
        first_key = list(adata.layers.keys())[0]
        return adata.layers[first_key]
    return None

def safe_read_h5ad(h5ad_path, tool_name=''):
    """Read H5AD file, handling non-standard formats from different tools.
    
    Returns (adata, error_msg). If error_msg is not None, reading failed.
    """
    import h5py
    # First check if it's a valid HDF5 file
    try:
        with h5py.File(h5ad_path, 'r') as f:
            keys = list(f.keys())
    except Exception as e:
        return None, f'not valid HDF5: {e}'
    
    # easySCF custom format: use its own loadH5 to read
    if 'X' not in keys and 'assay' in keys:
        try:
            from easySCFpy import loadH5
            adata = loadH5(h5ad_path)
            # For processed datasets, adata.X is normalized subset (e.g. 2700x2000),
            # while adata.raw.X is the full counts matrix (e.g. 2700x13714).
            # We need the full counts for fidelity comparison.
            try:
                if adata.raw is not None:
                    adata = ad.AnnData(X=adata.raw.X)
            except Exception:
                pass  # no raw, keep adata.X as-is (raw datasets)
            return adata, None
        except Exception as e:
            return None, f'easySCF loadH5 error: {e}'
    
    # Completely unrecognized format
    if 'X' not in keys and 'obs' not in keys:
        return None, f'non-standard H5 format, keys={keys}'
    
    # Standard AnnData read
    try:
        adata = ad.read_h5ad(h5ad_path)
        return adata, None
    except Exception as e:
        return None, str(e)


def compute_metrics(X1, X2, max_n=500000):
    """Compute 7 fidelity metrics between two expression matrices."""
    if X1.shape != X2.shape:
        return None, f'shape mismatch {X1.shape} vs {X2.shape}'
    total = X1.shape[0] * X1.shape[1]
    if total <= max_n:
        f1 = np.asarray(X1.todense() if sp.issparse(X1) else X1).flatten()
        f2 = np.asarray(X2.todense() if sp.issparse(X2) else X2).flatten()
    else:
        np.random.seed(42)
        ri = np.random.randint(0, X1.shape[0], max_n)
        ci = np.random.randint(0, X1.shape[1], max_n)
        if sp.issparse(X1):
            f1 = np.array([X1[r, c] for r, c in zip(ri, ci)]).flatten()
        else:
            f1 = X1[ri, ci].flatten()
        if sp.issparse(X2):
            f2 = np.array([X2[r, c] for r, c in zip(ri, ci)]).flatten()
        else:
            f2 = X2[ri, ci].flatten()
    if np.std(f1) == 0 or np.std(f2) == 0:
        pr = 1.0 if np.allclose(f1, f2) else 0.0
        sr = pr
    else:
        pr, _ = pearsonr(f1, f2)
        sr, _ = spearmanr(f1, f2)
    diff = np.abs(f1 - f2)
    mse = float(np.mean(diff**2))
    exact = np.sum(np.isclose(f1, f2, rtol=1e-7, atol=1e-10))
    return {
        'pearson_r': float(pr), 'spearman_rho': float(sr),
        'mse': mse, 'rmse': float(np.sqrt(mse)),
        'max_diff': float(np.max(diff)), 'mean_diff': float(np.mean(diff)),
        'exact_match_pct': float(exact / len(f1) * 100),
    }, None

def extract_error_reason(stderr_text):
    if not stderr_text:
        return 'unknown error'
    for line in stderr_text.split('\n'):
        line = line.strip()
        if line.startswith('Error') or 'error' in line.lower():
            return line[:120]
    lines = [l.strip() for l in stderr_text.strip().split('\n') if l.strip()]
    return lines[-1][:120] if lines else 'unknown error'

print('Helper functions defined')


Helper functions defined


## 4. CrossCell Reference Conversion (RDS -> H5AD)

Convert all 42 RDS datasets to H5AD using CrossCell as the reference standard.

In [4]:
cc_ref_results = []
print('=== CrossCell Reference Conversions ===')
for i, ds in enumerate(DATASETS):
    rds_path = str(DATA_DIR / ds['file'])
    h5ad_path = str(TMP / f"cc_{ds['test_id']}.h5ad")
    if not Path(rds_path).exists():
        print(f"  [{i+1}/{len(DATASETS)}] {ds['test_id']}: FILE NOT FOUND, skip")
        cc_ref_results.append({**ds, 'status': 'not_found'})
        continue
    # Skip if already converted
    if Path(h5ad_path).exists():
        print(f"  [{i+1}/{len(DATASETS)}] {ds['test_id']}: cached")
        cc_ref_results.append({**ds, 'status': 'success', 'h5ad_path': h5ad_path})
        continue
    ok, stderr = run_crosscell(rds_path, h5ad_path)
    status = 'success' if ok else 'failed'
    print(f"  [{i+1}/{len(DATASETS)}] {ds['test_id']}: {status}")
    entry = {**ds, 'status': status, 'h5ad_path': h5ad_path if ok else None}
    if not ok:
        entry['error'] = extract_error_reason(stderr)
    cc_ref_results.append(entry)

n_ok = sum(1 for r in cc_ref_results if r['status'] == 'success')
print(f'\nCrossCell reference: {n_ok}/{len(DATASETS)} converted')

=== CrossCell Reference Conversions ===
  [1/42] v4_pbmc3k_raw: cached
  [2/42] v4_pbmc3k_processed: cached
  [3/42] v5_pbmc3k_raw: cached
  [4/42] v5_pbmc3k_processed: cached
  [5/42] v4_stxKidney_raw: cached
  [6/42] v5_stxKidney_raw: cached
  [7/42] v4_celegans.embryo_raw: cached
  [8/42] v4_celegans.embryo_processed: cached
  [9/42] v5_celegans.embryo_raw: cached
  [10/42] v5_celegans.embryo_processed: cached
  [11/42] v4_cbmc_raw: cached
  [12/42] v4_cbmc_processed: cached
  [13/42] v5_cbmc_raw: cached
  [14/42] v5_cbmc_processed: cached
  [15/42] v4_ifnb_raw: cached
  [16/42] v4_ifnb_processed: cached
  [17/42] v5_ifnb_raw: cached
  [18/42] v5_ifnb_processed: cached
  [19/42] v4_panc8_raw: cached
  [20/42] v4_panc8_processed: cached
  [21/42] v5_panc8_raw: cached
  [22/42] v5_panc8_processed: cached
  [23/42] v4_thp1.eccite_raw: cached
  [24/42] v4_thp1.eccite_processed: cached
  [25/42] v5_thp1.eccite_raw: cached
  [26/42] v5_thp1.eccite_processed: cached
  [27/42] v4_bmcite_raw

## 5. CrossCell Roundtrip Fidelity (H5AD -> RDS -> H5AD)

Full roundtrip for each dataset, compare result against the first H5AD.

In [5]:
cc_roundtrip_results = []
print('=== CrossCell Roundtrip Fidelity ===')
for i, ref in enumerate(cc_ref_results):
    if ref['status'] != 'success':
        continue
    tid = ref['test_id']
    cc_h5ad = ref['h5ad_path']
    rt_rds = str(TMP / f'cc_rt_{tid}.rds')
    rt_h5ad = str(TMP / f'cc_rt_{tid}.h5ad')
    print(f"  [{i+1}/{len(DATASETS)}] {tid}...", end=' ')
    # H5AD -> RDS
    ok1, err1 = run_crosscell(cc_h5ad, rt_rds, 'seurat')
    if not ok1:
        print('H5AD->RDS FAILED'); continue
    # RDS -> H5AD
    ok2, err2 = run_crosscell(rt_rds, rt_h5ad, 'anndata')
    if not ok2:
        print('RDS->H5AD FAILED'); continue
    # Compare
    try:
        adata_ref = ad.read_h5ad(cc_h5ad)
        adata_rt = ad.read_h5ad(rt_h5ad)
        X_ref = get_X_matrix(adata_ref)
        X_rt = get_X_matrix(adata_rt)
        if X_ref is None or X_rt is None:
            print('X is None'); continue
        metrics, err = compute_metrics(X_ref, X_rt)
        if metrics:
            print(f"R={metrics['pearson_r']:.6f} MSE={metrics['mse']:.2e} Exact={metrics['exact_match_pct']:.1f}%")
            cc_roundtrip_results.append({
                'test_id': tid, 'tool': 'CrossCell', 'test_type': 'roundtrip',
                'dataset': ref['dataset'], 'n_cells': ref['n_cells'], 'n_genes': ref['n_genes'],
                **metrics
            })
        else:
            print(f'metrics error: {err}')
    except Exception as e:
        print(f'ERROR: {e}')

print(f'\nCrossCell roundtrip: {len(cc_roundtrip_results)} fidelity tests')


=== CrossCell Roundtrip Fidelity ===
  [1/42] v4_pbmc3k_raw... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [2/42] v4_pbmc3k_processed... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [3/42] v5_pbmc3k_raw... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [4/42] v5_pbmc3k_processed... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [5/42] v4_stxKidney_raw... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [6/42] v5_stxKidney_raw... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [7/42] v4_celegans.embryo_raw... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [8/42] v4_celegans.embryo_processed... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [9/42] v5_celegans.embryo_raw... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [10/42] v5_celegans.embryo_processed... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [11/42] v4_cbmc_raw... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [12/42] v4_cbmc_processed... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [13/42] v5_cbmc_raw... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [14/42] v5_cbmc_processed... R=1.000000 MSE=0.00e+00 Exact=100.0%
  [15/

## 6. Other Tools Fidelity (RDS -> H5AD vs CrossCell reference)

Each tool in a separate cell. Only tests datasets where the tool succeeded in robustness benchmark.

In [6]:
# Initialize shared results list for all other tools
other_tool_results = []
print('Initialized other_tool_results')


Initialized other_tool_results


### 6.1 Zellkonverter

In [7]:
tool = 'Zellkonverter'
other_tool_results = [r for r in other_tool_results if r['tool'] != tool]
print(f'=== {tool} ===')
tool_count = 0
for i, ref in enumerate(cc_ref_results):
    if ref['status'] != 'success':
        continue
    tid = ref['test_id']
    if not success_map.get(tool, {}).get(tid, False):
        continue
    rds_path = str(DATA_DIR / ref['file'])
    tool_h5ad = str(TMP / f'{tool}_{tid}.h5ad')
    cc_h5ad = ref['h5ad_path']
    print(f"  [{tool_count+1}] {tid}...", end=' ')
    try:
        ok, stderr = run_r_tool(rds_path, tool_h5ad, tool)
    except subprocess.TimeoutExpired:
        print('TIMEOUT'); continue
    except Exception as e:
        print(f'ERROR: {e}'); continue
    if not ok:
        print('FAILED')
        other_tool_results.append({
            'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
            'dataset': ref['dataset'], 'status': 'failed',
            'error': extract_error_reason(stderr),
        })
        tool_count += 1
        continue
    try:
        adata_cc, cc_err = safe_read_h5ad(cc_h5ad, 'CrossCell')
        if cc_err:
            print(f'CC read error: {cc_err}'); continue
        X_cc = get_X_matrix(adata_cc)
        if X_cc is None:
            print('CC X is None'); continue
        adata_tool, tool_err = safe_read_h5ad(tool_h5ad, tool)
        if tool_err:
            print(f'read error: {tool_err}')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'read_error',
                'error': tool_err[:200],
            })
            tool_count += 1
            continue
        X_tool = get_X_matrix(adata_tool)
        if X_tool is None:
            print('tool X is None (no X or layers)')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'no_matrix',
            })
            tool_count += 1
            continue
        if X_cc.shape != X_tool.shape:
            print(f'shape mismatch {X_cc.shape} vs {X_tool.shape}')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'shape_mismatch',
                'shape_cc': list(X_cc.shape), 'shape_tool': list(X_tool.shape),
            })
            tool_count += 1
            continue
        metrics, err = compute_metrics(X_cc, X_tool)
        if metrics:
            print(f"R={metrics['pearson_r']:.6f} MSE={metrics['mse']:.2e}")
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'success',
                'n_cells': ref['n_cells'], 'n_genes': ref['n_genes'],
                **metrics
            })
        else:
            print(f'metrics error: {err}')
    except Exception as e:
        print(f'read/compare ERROR: {e}')
        other_tool_results.append({
            'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
            'dataset': ref['dataset'], 'status': 'error', 'error': str(e)[:200],
        })
    tool_count += 1
n_zk = len([r for r in other_tool_results if r['tool'] == tool])
print(f'\n{tool}: {tool_count} datasets tested, {n_zk} results recorded')


=== Zellkonverter ===
  [1] v4_pbmc3k_raw... R=1.000000 MSE=0.00e+00
  [2] v4_pbmc3k_processed... R=1.000000 MSE=0.00e+00
  [3] v5_pbmc3k_raw... R=1.000000 MSE=0.00e+00
  [4] v5_pbmc3k_processed... R=1.000000 MSE=0.00e+00
  [5] v4_stxKidney_raw... R=1.000000 MSE=0.00e+00
  [6] v5_stxKidney_raw... R=1.000000 MSE=0.00e+00
  [7] v4_celegans.embryo_raw... R=1.000000 MSE=0.00e+00
  [8] v4_celegans.embryo_processed... R=1.000000 MSE=0.00e+00
  [9] v5_celegans.embryo_raw... R=1.000000 MSE=0.00e+00
  [10] v5_celegans.embryo_processed... R=1.000000 MSE=0.00e+00
  [11] v4_cbmc_raw... R=1.000000 MSE=0.00e+00
  [12] v4_cbmc_processed... R=1.000000 MSE=0.00e+00
  [13] v5_cbmc_raw... R=1.000000 MSE=0.00e+00
  [14] v5_cbmc_processed... R=1.000000 MSE=0.00e+00
  [15] v4_ifnb_raw... R=1.000000 MSE=0.00e+00
  [16] v4_ifnb_processed... R=1.000000 MSE=0.00e+00
  [17] v5_ifnb_raw... R=1.000000 MSE=0.00e+00
  [18] v5_ifnb_processed... R=1.000000 MSE=0.00e+00
  [19] v4_panc8_raw... R=1.000000 MSE=0.00e+00
  

### 6.2 anndataR

In [8]:
tool = 'anndataR'
other_tool_results = [r for r in other_tool_results if r['tool'] != tool]
print(f'=== {tool} ===')
tool_count = 0
for i, ref in enumerate(cc_ref_results):
    if ref['status'] != 'success':
        continue
    tid = ref['test_id']
    if not success_map.get(tool, {}).get(tid, False):
        continue
    rds_path = str(DATA_DIR / ref['file'])
    tool_h5ad = str(TMP / f'{tool}_{tid}.h5ad')
    cc_h5ad = ref['h5ad_path']
    print(f"  [{tool_count+1}] {tid}...", end=' ')
    try:
        ok, stderr = run_r_tool(rds_path, tool_h5ad, tool)
    except subprocess.TimeoutExpired:
        print('TIMEOUT'); continue
    except Exception as e:
        print(f'ERROR: {e}'); continue
    if not ok:
        print('FAILED')
        other_tool_results.append({
            'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
            'dataset': ref['dataset'], 'status': 'failed',
            'error': extract_error_reason(stderr),
        })
        tool_count += 1
        continue
    try:
        adata_cc, cc_err = safe_read_h5ad(cc_h5ad, 'CrossCell')
        if cc_err:
            print(f'CC read error: {cc_err}'); continue
        X_cc = get_X_matrix(adata_cc)
        if X_cc is None:
            print('CC X is None'); continue
        adata_tool, tool_err = safe_read_h5ad(tool_h5ad, tool)
        if tool_err:
            print(f'read error: {tool_err}')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'read_error',
                'error': tool_err[:200],
            })
            tool_count += 1
            continue
        X_tool = get_X_matrix(adata_tool)
        if X_tool is None:
            print('tool X is None (no X or layers)')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'no_matrix',
            })
            tool_count += 1
            continue
        if X_cc.shape != X_tool.shape:
            print(f'shape mismatch {X_cc.shape} vs {X_tool.shape}')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'shape_mismatch',
                'shape_cc': list(X_cc.shape), 'shape_tool': list(X_tool.shape),
            })
            tool_count += 1
            continue
        metrics, err = compute_metrics(X_cc, X_tool)
        if metrics:
            print(f"R={metrics['pearson_r']:.6f} MSE={metrics['mse']:.2e}")
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'success',
                'n_cells': ref['n_cells'], 'n_genes': ref['n_genes'],
                **metrics
            })
        else:
            print(f'metrics error: {err}')
    except Exception as e:
        print(f'read/compare ERROR: {e}')
        other_tool_results.append({
            'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
            'dataset': ref['dataset'], 'status': 'error', 'error': str(e)[:200],
        })
    tool_count += 1
n_ar = len([r for r in other_tool_results if r['tool'] == tool])
print(f'\n{tool}: {tool_count} datasets tested, {n_ar} results recorded')


=== anndataR ===
  [1] v4_pbmc3k_raw... R=1.000000 MSE=0.00e+00
  [2] v4_pbmc3k_processed... R=1.000000 MSE=0.00e+00
  [3] v5_pbmc3k_raw... R=1.000000 MSE=0.00e+00
  [4] v5_pbmc3k_processed... R=1.000000 MSE=0.00e+00
  [5] v4_stxKidney_raw... R=1.000000 MSE=0.00e+00
  [6] v5_stxKidney_raw... R=1.000000 MSE=0.00e+00
  [7] v4_celegans.embryo_raw... R=1.000000 MSE=0.00e+00
  [8] v4_celegans.embryo_processed... R=1.000000 MSE=0.00e+00
  [9] v5_celegans.embryo_raw... R=1.000000 MSE=0.00e+00
  [10] v5_celegans.embryo_processed... R=1.000000 MSE=0.00e+00
  [11] v4_cbmc_raw... R=1.000000 MSE=0.00e+00
  [12] v4_cbmc_processed... R=1.000000 MSE=0.00e+00
  [13] v5_cbmc_raw... R=1.000000 MSE=0.00e+00
  [14] v5_cbmc_processed... R=1.000000 MSE=0.00e+00
  [15] v4_ifnb_raw... R=1.000000 MSE=0.00e+00
  [16] v4_ifnb_processed... R=1.000000 MSE=0.00e+00
  [17] v5_ifnb_raw... R=1.000000 MSE=0.00e+00
  [18] v5_ifnb_processed... R=1.000000 MSE=0.00e+00
  [19] v4_panc8_raw... R=1.000000 MSE=0.00e+00
  [20] 

### 6.3 convert2anndata

In [9]:
tool = 'convert2anndata'
other_tool_results = [r for r in other_tool_results if r['tool'] != tool]
print(f'=== {tool} ===')
tool_count = 0
for i, ref in enumerate(cc_ref_results):
    if ref['status'] != 'success':
        continue
    tid = ref['test_id']
    if not success_map.get(tool, {}).get(tid, False):
        continue
    rds_path = str(DATA_DIR / ref['file'])
    tool_h5ad = str(TMP / f'{tool}_{tid}.h5ad')
    cc_h5ad = ref['h5ad_path']
    print(f"  [{tool_count+1}] {tid}...", end=' ')
    try:
        ok, stderr = run_r_tool(rds_path, tool_h5ad, tool)
    except subprocess.TimeoutExpired:
        print('TIMEOUT'); continue
    except Exception as e:
        print(f'ERROR: {e}'); continue
    if not ok:
        print('FAILED')
        other_tool_results.append({
            'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
            'dataset': ref['dataset'], 'status': 'failed',
            'error': extract_error_reason(stderr),
        })
        tool_count += 1
        continue
    try:
        adata_cc, cc_err = safe_read_h5ad(cc_h5ad, 'CrossCell')
        if cc_err:
            print(f'CC read error: {cc_err}'); continue
        X_cc = get_X_matrix(adata_cc)
        if X_cc is None:
            print('CC X is None'); continue
        adata_tool, tool_err = safe_read_h5ad(tool_h5ad, tool)
        if tool_err:
            print(f'read error: {tool_err}')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'read_error',
                'error': tool_err[:200],
            })
            tool_count += 1
            continue
        X_tool = get_X_matrix(adata_tool)
        if X_tool is None:
            print('tool X is None (no X or layers)')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'no_matrix',
            })
            tool_count += 1
            continue
        if X_cc.shape != X_tool.shape:
            print(f'shape mismatch {X_cc.shape} vs {X_tool.shape}')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'shape_mismatch',
                'shape_cc': list(X_cc.shape), 'shape_tool': list(X_tool.shape),
            })
            tool_count += 1
            continue
        metrics, err = compute_metrics(X_cc, X_tool)
        if metrics:
            print(f"R={metrics['pearson_r']:.6f} MSE={metrics['mse']:.2e}")
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'success',
                'n_cells': ref['n_cells'], 'n_genes': ref['n_genes'],
                **metrics
            })
        else:
            print(f'metrics error: {err}')
    except Exception as e:
        print(f'read/compare ERROR: {e}')
        other_tool_results.append({
            'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
            'dataset': ref['dataset'], 'status': 'error', 'error': str(e)[:200],
        })
    tool_count += 1
n_c2a = len([r for r in other_tool_results if r['tool'] == tool])
print(f'\n{tool}: {tool_count} datasets tested, {n_c2a} results recorded')


=== convert2anndata ===
  [1] v5_pbmc3k_raw... R=1.000000 MSE=0.00e+00
  [2] v5_pbmc3k_processed... R=1.000000 MSE=0.00e+00
  [3] v5_stxKidney_raw... R=1.000000 MSE=0.00e+00
  [4] v5_celegans.embryo_raw... R=1.000000 MSE=0.00e+00
  [5] v5_celegans.embryo_processed... R=1.000000 MSE=0.00e+00
  [6] v5_cbmc_raw... R=1.000000 MSE=0.00e+00
  [7] v5_cbmc_processed... R=1.000000 MSE=0.00e+00
  [8] v5_ifnb_raw... R=1.000000 MSE=0.00e+00
  [9] v5_ifnb_processed... R=1.000000 MSE=0.00e+00
  [10] v5_panc8_raw... R=1.000000 MSE=1.05e-11
  [11] v5_panc8_processed... R=1.000000 MSE=1.05e-11
  [12] v5_thp1.eccite_raw... R=1.000000 MSE=0.00e+00
  [13] v5_thp1.eccite_processed... R=1.000000 MSE=0.00e+00
  [14] v5_bmcite_raw... R=1.000000 MSE=0.00e+00
  [15] v5_bmcite_processed... R=1.000000 MSE=0.00e+00
  [16] v5_pbmcsca_raw... R=1.000000 MSE=0.00e+00
  [17] v5_pbmcsca_processed... R=1.000000 MSE=0.00e+00
  [18] v5_hcabm40k_raw... R=1.000000 MSE=0.00e+00
  [19] v5_hcabm40k_processed... R=1.000000 MSE=0

### 6.4 easySCF

In [10]:
tool = 'easySCF'
other_tool_results = [r for r in other_tool_results if r['tool'] != tool]
print(f'=== {tool} ===')
tool_count = 0
for i, ref in enumerate(cc_ref_results):
    if ref['status'] != 'success':
        continue
    tid = ref['test_id']
    if not success_map.get(tool, {}).get(tid, False):
        continue
    rds_path = str(DATA_DIR / ref['file'])
    tool_h5ad = str(TMP / f'{tool}_{tid}.h5ad')
    cc_h5ad = ref['h5ad_path']
    print(f"  [{tool_count+1}] {tid}...", end=' ')
    try:
        ok, stderr = run_r_tool(rds_path, tool_h5ad, tool)
    except subprocess.TimeoutExpired:
        print('TIMEOUT'); continue
    except Exception as e:
        print(f'ERROR: {e}'); continue
    if not ok:
        print('FAILED')
        other_tool_results.append({
            'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
            'dataset': ref['dataset'], 'status': 'failed',
            'error': extract_error_reason(stderr),
        })
        tool_count += 1
        continue
    try:
        adata_cc, cc_err = safe_read_h5ad(cc_h5ad, 'CrossCell')
        if cc_err:
            print(f'CC read error: {cc_err}'); continue
        X_cc = get_X_matrix(adata_cc)
        if X_cc is None:
            print('CC X is None'); continue
        adata_tool, tool_err = safe_read_h5ad(tool_h5ad, tool)
        if tool_err:
            print(f'read error: {tool_err}')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'read_error',
                'error': tool_err[:200],
            })
            tool_count += 1
            continue
        X_tool = get_X_matrix(adata_tool)
        if X_tool is None:
            print('tool X is None (no X or layers)')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'no_matrix',
            })
            tool_count += 1
            continue
        if X_cc.shape != X_tool.shape:
            print(f'shape mismatch {X_cc.shape} vs {X_tool.shape}')
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'shape_mismatch',
                'shape_cc': list(X_cc.shape), 'shape_tool': list(X_tool.shape),
            })
            tool_count += 1
            continue
        metrics, err = compute_metrics(X_cc, X_tool)
        if metrics:
            print(f"R={metrics['pearson_r']:.6f} MSE={metrics['mse']:.2e}")
            other_tool_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
                'dataset': ref['dataset'], 'status': 'success',
                'n_cells': ref['n_cells'], 'n_genes': ref['n_genes'],
                **metrics
            })
        else:
            print(f'metrics error: {err}')
    except Exception as e:
        print(f'read/compare ERROR: {e}')
        other_tool_results.append({
            'test_id': tid, 'tool': tool, 'test_type': 'cross_tool',
            'dataset': ref['dataset'], 'status': 'error', 'error': str(e)[:200],
        })
    tool_count += 1
n_easy = len([r for r in other_tool_results if r['tool'] == tool])
print(f'\n{tool}: {tool_count} datasets tested, {n_easy} results recorded')


=== easySCF ===
  [1] v4_pbmc3k_raw... Reading X data
Reading reductions data
Reading graphs data
Reading commands data
Reading images data
R=1.000000 MSE=0.00e+00
  [2] v4_pbmc3k_processed... Reading X data
Reading dense matrix
Reading reductions data
Reading graphs data
Reading commands data
Reading images data
R=1.000000 MSE=0.00e+00
  [3] v5_pbmc3k_raw... Reading X data
Reading reductions data
Reading graphs data
Reading commands data
Reading images data
R=1.000000 MSE=0.00e+00
  [4] v5_pbmc3k_processed... Reading X data
Reading dense matrix
Reading reductions data
Reading graphs data
Reading commands data
Reading images data
R=1.000000 MSE=0.00e+00
  [5] v4_stxKidney_raw... read error: non-standard H5 format, keys=[]
  [6] v5_stxKidney_raw... read error: non-standard H5 format, keys=[]
  [7] v4_celegans.embryo_raw... Reading X data
Reading reductions data
Reading graphs data
Reading commands data
Reading images data
R=1.000000 MSE=0.00e+00
  [8] v4_celegans.embryo_processed... Rea

## 7. Save Results

In [11]:
all_results = {
    'crosscell_roundtrip': cc_roundtrip_results,
    'cross_tool': other_tool_results,
    'metadata': {
        'n_datasets': len(DATASETS),
        'tools': ALL_TOOLS,
        'metrics': ['pearson_r', 'spearman_rho', 'mse', 'rmse', 'max_diff', 'mean_diff', 'exact_match_pct'],
    }
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Saved to {OUTPUT_FILE}')

# Summary
print(f'\n=== Summary ===')
print(f'CrossCell roundtrip: {len(cc_roundtrip_results)} datasets')
if cc_roundtrip_results:
    avg_r = np.mean([r['pearson_r'] for r in cc_roundtrip_results])
    avg_mse = np.mean([r['mse'] for r in cc_roundtrip_results])
    avg_exact = np.mean([r['exact_match_pct'] for r in cc_roundtrip_results])
    print(f'  Avg Pearson R: {avg_r:.6f}')
    print(f'  Avg MSE: {avg_mse:.2e}')
    print(f'  Avg Exact Match: {avg_exact:.1f}%')

for tool in OTHER_TOOLS:
    tool_res = [r for r in other_tool_results if r['tool'] == tool and r.get('status') == 'success']
    n_fail = len([r for r in other_tool_results if r['tool'] == tool and r.get('status') != 'success'])
    print(f'\n{tool}: {len(tool_res)} success, {n_fail} failed/skipped')
    if tool_res:
        avg_r = np.mean([r['pearson_r'] for r in tool_res])
        avg_mse = np.mean([r['mse'] for r in tool_res])
        print(f'  Avg Pearson R: {avg_r:.6f}')
        print(f'  Avg MSE: {avg_mse:.2e}')

Saved to /benchmark/results/multi_tool_fidelity.json

=== Summary ===
CrossCell roundtrip: 42 datasets
  Avg Pearson R: 1.000000
  Avg MSE: 0.00e+00
  Avg Exact Match: 100.0%

Zellkonverter: 42 success, 0 failed/skipped
  Avg Pearson R: 1.000000
  Avg MSE: 0.00e+00

anndataR: 42 success, 0 failed/skipped
  Avg Pearson R: 1.000000
  Avg MSE: 0.00e+00

convert2anndata: 19 success, 0 failed/skipped
  Avg Pearson R: 1.000000
  Avg MSE: 1.11e-12

easySCF: 36 success, 6 failed/skipped
  Avg Pearson R: 1.000000
  Avg MSE: 0.00e+00
